## KV Cache

A KV Cache (Key-Value Cache) is a crucial optimization in Large Language Models (LLMs) that speeds up text generation by storing intermediate attention calculations. Instead of recalculating the context history from scratch for every single word generated, the model reuses past Key and Value vectors, drastically reducing latency and computational overhead.

Let's apply the same technique for our attention based model which we were training in the previous notebook: `6_Attention_is_All_You_Need.ipynb`

In [1]:
# %pip install -U datasets

from collections import Counter
from types import SimpleNamespace
from typing import Iterable, List

import re

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import numpy as np

import random
import math
import time

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [3]:
SOURCE_LANGUAGE = "de"
TARGET_LANGUAGE = "en"

DATASET_NAME = "bentrevett/multi30k"
SPLIT_NAMES = {
    "train": "train",
    "valid": "validation",
    "test": "test",
}

In [4]:
def simple_tokenize(text: str) -> List[str]:
    return re.findall(r"\w+|[^\w\s]", text.lower(), flags=re.UNICODE)


def tokenize_de(text):
    """Tokenizes German text from a string into a list of strings."""
    return simple_tokenize(text)


def tokenize_en(text):
    """Tokenizes English text from a string into a list of strings."""
    return simple_tokenize(text)


class DefaultIndexDict(dict):
    def __init__(self, default_index: int):
        super().__init__()
        self.default_index = default_index

    def __missing__(self, key):
        return self.default_index


class Vocab:
    def __init__(self, token_lists: Iterable[List[str]], min_freq: int, specials: List[str]):
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)

        self.itos = list(specials)
        seen = set(self.itos)
        for token, freq in sorted(counter.items(), key=lambda item: (-item[1], item[0])):
            if freq >= min_freq and token not in seen:
                self.itos.append(token)
                seen.add(token)

        self.stoi = DefaultIndexDict(self.itos.index("<unk>"))
        self.stoi.update({token: index for index, token in enumerate(self.itos)})

    def __len__(self):
        return len(self.itos)


class Language:
    def __init__(self, tokenize, attribute: str, init_token="<bos>", eos_token="<eos>", pad_token="<pad>", unk_token="<unk>"):
        self.tokenize = tokenize
        self.attribute = attribute
        self.init_token = init_token
        self.eos_token = eos_token
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.vocab = None

    def preprocess(self, text):
        if isinstance(text, str):
            return self.tokenize(text)
        return [token.lower() for token in text]

    def build_vocab(self, data, min_freq: int):
        specials = [self.unk_token, self.pad_token, self.init_token, self.eos_token]
        self.vocab = Vocab((getattr(example, self.attribute) for example in data), min_freq, specials)


class TranslationExample:
    def __init__(self, src: List[str], trg: List[str]):
        self.src = src
        self.trg = trg


class TranslationDataset:
    def __init__(self, examples: List[TranslationExample]):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

    def __iter__(self):
        return iter(self.examples)


def get_language_pair(example):
    if SOURCE_LANGUAGE in example and TARGET_LANGUAGE in example:
        return example[SOURCE_LANGUAGE], example[TARGET_LANGUAGE]

    translation = example.get("translation")
    if translation is not None:
        return translation[SOURCE_LANGUAGE], translation[TARGET_LANGUAGE]

    raise KeyError(f"Example has unsupported columns: {list(example.keys())}")


def load_multi30k_split(split: str):
    dataset = load_dataset(DATASET_NAME, split=SPLIT_NAMES[split])
    examples = []
    for example in dataset:
        source, target = get_language_pair(example)
        examples.append(TranslationExample(tokenize_de(source), tokenize_en(target)))
    return TranslationDataset(examples)

In [5]:
SRC = Language(tokenize_de, attribute="src")
TRG = Language(tokenize_en, attribute="trg")

train_data = load_multi30k_split("train")
valid_data = load_multi30k_split("valid")
test_data = load_multi30k_split("test")

SRC.build_vocab(train_data, min_freq = 2)
TRG.build_vocab(train_data, min_freq = 2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
BATCH_SIZE = 128


def numericalize(tokens: List[str], language: Language):
    return [
        language.vocab.stoi[language.init_token],
        *[language.vocab.stoi[token] for token in tokens],
        language.vocab.stoi[language.eos_token],
    ]


def collate_fn(batch):
    source_batch = [
        torch.tensor(numericalize(example.src, SRC), dtype=torch.long)
        for example in batch
    ]
    target_batch = [
        torch.tensor(numericalize(example.trg, TRG), dtype=torch.long)
        for example in batch
    ]

    source_batch = pad_sequence(
        source_batch,
        batch_first=True,
        padding_value=SRC.vocab.stoi[SRC.pad_token],
    )
    target_batch = pad_sequence(
        target_batch,
        batch_first=True,
        padding_value=TRG.vocab.stoi[TRG.pad_token],
    )
    return SimpleNamespace(src=source_batch.to(device), trg=target_batch.to(device))


train_iterator = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)
valid_iterator = DataLoader(
    valid_data,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
)
test_iterator = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    collate_fn=collate_fn,
)

## Seq2Seq Model

The model should be adjusted in order to handle the KV cache, we are going to store and collect this data for decoder only since the encoder cache is not required and run only once, unlike the decode which running with complexity $O(n^2)$. Cache work per step, processes only 1 token each time $O(n)$


In [8]:
class Encoder(nn.Module):
    def __init__(self,
                 input_dim,
                 hid_dim,
                 n_layers,
                 n_heads,
                 pf_dim,
                 dropout,
                 device,
                 max_length = 100):
        super().__init__()

        self.device = device

        self.tok_embedding = nn.Embedding(input_dim, hid_dim)
        self.pos_embedding = nn.Embedding(max_length, hid_dim)

        self.layers = nn.ModuleList([EncoderLayer(hid_dim,
                                                  n_heads,
                                                  pf_dim,
                                                  dropout,
                                                  device)
                                     for _ in range(n_layers)])

        self.dropout = nn.Dropout(dropout)

        self.scale = torch.sqrt(torch.FloatTensor([hid_dim])).to(device)

    def forward(self, src, src_mask):

        #src = [batch size, src len]
        #src_mask = [batch size, 1, 1, src len]

        batch_size = src.shape[0]
        src_len = src.shape[1]

        pos = torch.arange(0, src_len).unsqueeze(0).repeat(batch_size, 1).to(self.device)

        #pos = [batch size, src len]

        src = self.dropout((self.tok_embedding(src) * self.scale) + self.pos_embedding(pos))

        #src = [batch size, src len, hid dim]

        for layer in self.layers:
            src = layer(src, src_mask)

        #src = [batch size, src len, hid dim]

        return src

In [95]:
class MultiHeadAttentionLayer(nn.Module):
    def __init__(self, hid_dim, n_heads, dropout, device):
        super().__init__()

        assert hid_dim % n_heads == 0

        self.hid_dim = hid_dim
        self.n_heads = n_heads
        self.head_dim = hid_dim // n_heads

        self.fc_q = nn.Linear(hid_dim, hid_dim)
        self.fc_k = nn.Linear(hid_dim, hid_dim)
        self.fc_v = nn.Linear(hid_dim, hid_dim)

        self.fc_o = nn.Linear(hid_dim, hid_dim)

        self.dropout = nn.Dropout(dropout)

        self.scale = torch.sqrt(torch.FloatTensor([self.head_dim])).to(device)


    def forward(self, query, key, value, mask=None, past_kv=None): # cache is passed here

        batch_size = query.shape[0]

        #query = [batch size, query len, hid dim]
        #key = [batch size, key len, hid dim]
        #value = [batch size, value len, hid dim]

        Q = self.fc_q(query)
        K = self.fc_k(key)
        V = self.fc_v(value)

        #Q = [batch size, query len, hid dim]
        #K = [batch size, key len, hid dim]
        #V = [batch size, value len, hid dim]

        # It is not used in the training pipeline so it does not speed up traning
        # only for inferece
        if past_kv is not None:
            past_k, past_v = past_kv
            K = torch.cat([past_k, K], dim=1)
            V = torch.cat([past_v, V], dim=1)

        current_kv = (K, V)

        Q = Q.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        K = K.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        V = V.view(batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)

        #Q = [batch size, n heads, query len, head dim]
        #K = [batch size, n heads, key len, head dim]
        #V = [batch size, n heads, value len, head dim]

        energy = torch.matmul(Q, K.permute(0, 1, 3, 2)) / self.scale

        #energy = [batch size, n heads, query len, key len]

        if mask is not None:
            energy = energy.masked_fill(mask == 0, -1e10)

        attention = torch.softmax(energy, dim = -1)

        #attention = [batch size, n heads, query len, key len]

        x = torch.matmul(self.dropout(attention), V)

        #x = [batch size, n heads, query len, head dim]

        x = x.permute(0, 2, 1, 3).contiguous()

        #x = [batch size, query len, n heads, head dim]

        x = x.view(batch_size, -1, self.hid_dim)

        #x = [batch size, query len, hid dim]

        x = self.fc_o(x)

        #x = [batch size, query len, hid dim]

        return x, attention, current_kv

In [96]:
class EncoderLayer(nn.Module):
    def __init__(self,
                 hid_dim,
                 n_heads,
                 pf_dim,
                 dropout,
                 device):
        super().__init__()

        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout, device)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim,
                                                                     pf_dim,
                                                                     dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask, past_kvs = None):

        #src = [batch size, src len, hid dim]
        #src_mask = [batch size, 1, 1, src len]

        #self attention
        _src, _, _ = self.self_attention(src, src, src, src_mask, past_kvs)

        #dropout, residual connection and layer norm
        src = self.self_attn_layer_norm(src + self.dropout(_src))

        #src = [batch size, src len, hid dim]

        #positionwise feedforward
        _src = self.positionwise_feedforward(src)

        #dropout, residual and layer norm
        src = self.ff_layer_norm(src + self.dropout(_src))

        #src = [batch size, src len, hid dim]

        return src

In [97]:
class PositionwiseFeedforwardLayer(nn.Module):
    def __init__(self, hid_dim, pf_dim, dropout):
        super().__init__()

        self.fc_1 = nn.Linear(hid_dim, pf_dim)
        self.fc_2 = nn.Linear(pf_dim, hid_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        #x = [batch size, seq len, hid dim]

        x = self.dropout(torch.relu(self.fc_1(x)))

        #x = [batch size, seq len, pf dim]

        x = self.fc_2(x)

        #x = [batch size, seq len, hid dim]

        return x

In [98]:
class DecoderLayer(nn.Module):
    def __init__(self,
                 hid_dim,
                 n_heads,
                 pf_dim,
                 dropout,
                 device):
        super().__init__()

        self.self_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.enc_attn_layer_norm = nn.LayerNorm(hid_dim)
        self.ff_layer_norm = nn.LayerNorm(hid_dim)
        self.self_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout, device)
        self.encoder_attention = MultiHeadAttentionLayer(hid_dim, n_heads, dropout, device)
        self.positionwise_feedforward = PositionwiseFeedforwardLayer(hid_dim,
                                                                     pf_dim,
                                                                     dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, trg, enc_src, trg_mask, src_mask, past_kv=None):

        #trg = [batch size, trg len, hid dim]
        #enc_src = [batch size, src len, hid dim]
        #trg_mask = [batch size, 1, trg len, trg len]
        #src_mask = [batch size, 1, 1, src len]

        #self attention
        _trg, self_attn, new_kv = self.self_attention(trg, trg, trg, trg_mask, past_kv)

        #dropout, residual connection and layer norm
        trg = self.self_attn_layer_norm(trg + self.dropout(_trg))

        #trg = [batch size, trg len, hid dim]

        #encoder attention
        #cross-attention: no cache needed, enc_src is static
        _trg, attention, _ = self.encoder_attention(trg, enc_src, enc_src, src_mask)

        #dropout, residual connection and layer norm
        trg = self.enc_attn_layer_norm(trg + self.dropout(_trg))

        #trg = [batch size, trg len, hid dim]

        #positionwise feedforward
        _trg = self.positionwise_feedforward(trg)

        #dropout, residual and layer norm
        trg = self.ff_layer_norm(trg + self.dropout(_trg))

        #trg = [batch size, trg len, hid dim]
        #attention = [batch size, n heads, trg len, src len]

        return trg, attention, new_kv

In [112]:
class Decoder(nn.Module):
    def __init__(self,
                 output_dim,
                 hid_dim,
                 n_layers,
                 n_heads,
                 pf_dim,
                 dropout,
                 device,
                 max_length = 100):
        super().__init__()

        self.device = device

        self.tok_embedding = nn.Embedding(output_dim, hid_dim)
        self.pos_embedding = nn.Embedding(max_length, hid_dim)
        self.n_layers = n_layers
        self.layers = nn.ModuleList([DecoderLayer(hid_dim,
                                                  n_heads,
                                                  pf_dim,
                                                  dropout,
                                                  device)
                                     for _ in range(n_layers)])

        self.fc_out = nn.Linear(hid_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

        self.scale = torch.sqrt(torch.FloatTensor([hid_dim])).to(device)

    def forward(self, trg, enc_src, trg_mask, src_mask, past_kvs=None):

        #trg = [batch size, trg len]
        #enc_src = [batch size, src len, hid dim]
        #trg_mask = [batch size, 1, trg len, trg len]
        #src_mask = [batch size, 1, 1, src len]
        if past_kvs is None:
            past_kvs = [None] * self.n_layers

        batch_size = trg.shape[0]
        trg_len = trg.shape[1]


        past_len = past_kvs[0][0].shape[1] if past_kvs[0] is not None else 0
        #Offset positions by cached length so token positions are correct
        pos = torch.arange(past_len, past_len + trg_len)\
                   .unsqueeze(0)\
                   .repeat(batch_size, 1)\
                   .to(self.device)


        #pos = [batch size, trg len]

        trg = self.dropout((self.tok_embedding(trg) * self.scale) + self.pos_embedding(pos))

        #trg = [batch size, trg len, hid dim]
        new_kvs = []
        for layer, past_kv in zip(self.layers, past_kvs):
            trg, attention, new_kv = layer(trg, enc_src, trg_mask, src_mask, past_kv)
            new_kvs.append(new_kv)

        #trg = [batch size, trg len, hid dim]
        #attention = [batch size, n heads, trg len, src len]

        output = self.fc_out(trg)

        #output = [batch size, trg len, output dim]

        return output, attention, new_kvs

In [113]:
class Seq2Seq(nn.Module):
    def __init__(self,
                 encoder,
                 decoder,
                 src_pad_idx,
                 trg_pad_idx,
                 device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.trg_pad_idx = trg_pad_idx
        self.device = device

    def make_src_mask(self, src):

        #src = [batch size, src len]

        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)

        #src_mask = [batch size, 1, 1, src len]

        return src_mask

    def make_trg_mask(self, trg):

        #trg = [batch size, trg len]

        trg_pad_mask = (trg != self.trg_pad_idx).unsqueeze(1).unsqueeze(2)

        #trg_pad_mask = [batch size, 1, 1, trg len]

        trg_len = trg.shape[1]

        trg_sub_mask = torch.tril(torch.ones((trg_len, trg_len), device = self.device)).bool()

        #trg_sub_mask = [trg len, trg len]

        trg_mask = trg_pad_mask & trg_sub_mask

        #trg_mask = [batch size, 1, trg len, trg len]

        return trg_mask

    def forward(self, src, trg):

        #src = [batch size, src len]
        #trg = [batch size, trg len]

        src_mask = self.make_src_mask(src)
        trg_mask = self.make_trg_mask(trg)

        #src_mask = [batch size, 1, 1, src len]
        #trg_mask = [batch size, 1, trg len, trg len]

        enc_src = self.encoder(src, src_mask)

        #enc_src = [batch size, src len, hid dim]

        output, attention, cache = self.decoder(trg, enc_src, trg_mask, src_mask)

        #output = [batch size, trg len, output dim]
        #attention = [batch size, n heads, trg len, src len]

        return output, attention, cache

In [114]:
INPUT_DIM = len(SRC.vocab)
OUTPUT_DIM = len(TRG.vocab)
HID_DIM = 256
ENC_LAYERS = 3
DEC_LAYERS = 3
ENC_HEADS = 8
DEC_HEADS = 8
ENC_PF_DIM = 512
DEC_PF_DIM = 512
ENC_DROPOUT = 0.1
DEC_DROPOUT = 0.1

enc = Encoder(INPUT_DIM,
              HID_DIM,
              ENC_LAYERS,
              ENC_HEADS,
              ENC_PF_DIM,
              ENC_DROPOUT,
              device)

dec = Decoder(OUTPUT_DIM,
              HID_DIM,
              DEC_LAYERS,
              DEC_HEADS,
              DEC_PF_DIM,
              DEC_DROPOUT,
              device)

In [115]:
SRC_PAD_IDX = SRC.vocab.stoi[SRC.pad_token]
TRG_PAD_IDX = TRG.vocab.stoi[TRG.pad_token]

model = Seq2Seq(enc, dec, SRC_PAD_IDX, TRG_PAD_IDX, device).to(device)

In [110]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 9,048,330 trainable parameters


In [104]:
def initialize_weights(m):
    if hasattr(m, 'weight') and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)

In [71]:
LEARNING_RATE = 0.0005

optimizer = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index = TRG_PAD_IDX)

In [117]:
def train(model, iterator, optimizer, criterion, clip):

    model.train()

    epoch_loss = 0

    for i, batch in enumerate(iterator):

        src = batch.src
        trg = batch.trg

        optimizer.zero_grad()

        output, _, cache = model(src, trg[:,:-1])

        #output = [batch size, trg len - 1, output dim]
        #trg = [batch size, trg len]

        output_dim = output.shape[-1]

        output = output.contiguous().view(-1, output_dim)
        trg = trg[:,1:].contiguous().view(-1)

        #output = [batch size * trg len - 1, output dim]
        #trg = [batch size * trg len - 1]

        loss = criterion(output, trg)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [118]:
def evaluate(model, iterator, criterion):

    model.eval()

    epoch_loss = 0

    with torch.no_grad():

        for i, batch in enumerate(iterator):

            src = batch.src
            trg = batch.trg

            output, _, cache = model(src, trg[:,:-1])

            #output = [batch size, trg len - 1, output dim]
            #trg = [batch size, trg len]

            output_dim = output.shape[-1]

            output = output.contiguous().view(-1, output_dim)
            trg = trg[:,1:].contiguous().view(-1)

            #output = [batch size * trg len - 1, output dim]
            #trg = [batch size * trg len - 1]

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [119]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [75]:
N_EPOCHS = 10
CLIP = 1

best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):

    start_time = time.time()

    train_loss = train(model, train_iterator, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, valid_iterator, criterion)

    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'tut6-model.pt')

    print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train PPL: {math.exp(train_loss):7.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):7.3f}')

Epoch: 01 | Time: 0m 15s
	Train Loss: 4.115 | Train PPL:  61.269
	 Val. Loss: 3.159 |  Val. PPL:  23.549
Epoch: 02 | Time: 0m 15s
	Train Loss: 3.028 | Train PPL:  20.654
	 Val. Loss: 2.676 |  Val. PPL:  14.522
Epoch: 03 | Time: 0m 15s
	Train Loss: 2.613 | Train PPL:  13.642
	 Val. Loss: 2.397 |  Val. PPL:  10.987
Epoch: 04 | Time: 0m 14s
	Train Loss: 2.333 | Train PPL:  10.308
	 Val. Loss: 2.226 |  Val. PPL:   9.262
Epoch: 05 | Time: 0m 15s
	Train Loss: 2.113 | Train PPL:   8.276
	 Val. Loss: 2.105 |  Val. PPL:   8.211
Epoch: 06 | Time: 0m 14s
	Train Loss: 1.938 | Train PPL:   6.948
	 Val. Loss: 2.009 |  Val. PPL:   7.453
Epoch: 07 | Time: 0m 14s
	Train Loss: 1.790 | Train PPL:   5.991
	 Val. Loss: 1.950 |  Val. PPL:   7.025
Epoch: 08 | Time: 0m 15s
	Train Loss: 1.662 | Train PPL:   5.272
	 Val. Loss: 1.915 |  Val. PPL:   6.789
Epoch: 09 | Time: 0m 15s
	Train Loss: 1.544 | Train PPL:   4.684
	 Val. Loss: 1.893 |  Val. PPL:   6.637
Epoch: 10 | Time: 0m 15s
	Train Loss: 1.448 | Train PPL

In [120]:
model.load_state_dict(torch.load('tut6-model.pt'))

test_loss = evaluate(model, test_iterator, criterion)

print(f'| Test Loss: {test_loss:.3f} | Test PPL: {math.exp(test_loss):7.3f} |')

| Test Loss: 1.899 | Test PPL:   6.677 |


In [121]:
def translate_sentence(sentence, src_field, trg_field, model, device, max_len=50):
    model.eval()

    tokens = src_field.preprocess(sentence)
    tokens = [src_field.init_token] + tokens + [src_field.eos_token]
    src_indexes = [src_field.vocab.stoi[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src_tensor)

    with torch.no_grad():
        #Encoder runs once - reused every decoding step
        enc_src = model.encoder(src_tensor, src_mask)

    trg_indexes = [trg_field.vocab.stoi[trg_field.init_token]]

    past_kvs = None  #Start with empty cache

    for i in range(max_len):
        #Feed only the last token (cache handles the rest)
        last_token = torch.LongTensor([trg_indexes[-1]]).unsqueeze(0).to(device)

        # Single token mask: no padding, no future masking needed
        trg_mask = model.make_trg_mask(last_token)

        with torch.no_grad():
            #Pass and receive updated cache each step
            output, attention, past_kvs = model.decoder(
                last_token, enc_src, trg_mask, src_mask, past_kvs
            )

        pred_token = output.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)

        if pred_token == trg_field.vocab.stoi[trg_field.eos_token]:
            break

    trg_tokens = [trg_field.vocab.itos[i] for i in trg_indexes]
    return trg_tokens[1:], attention

In [122]:
def display_attention(sentence, translation, attention, n_heads = 8, n_rows = 4, n_cols = 2):

    assert n_rows * n_cols == n_heads

    fig = plt.figure(figsize=(15,25))

    for i in range(n_heads):

        ax = fig.add_subplot(n_rows, n_cols, i+1)

        _attention = attention.squeeze(0)[i].cpu().detach().numpy()

        cax = ax.matshow(_attention, cmap='bone')

        ax.tick_params(labelsize=12)
        ax.set_xticklabels(['']+['<bos>']+[t.lower() for t in sentence]+['<eos>'],
                           rotation=45)
        ax.set_yticklabels(['']+translation)

        ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
        ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    plt.show()
    plt.close()

In [123]:
example_idx = 8

src = train_data.examples[example_idx].src
trg = train_data.examples[example_idx].trg

print(f'src = {src}')
print(f'trg = {trg}')

src = ['eine', 'frau', 'mit', 'einer', 'großen', 'geldbörse', 'geht', 'an', 'einem', 'tor', 'vorbei', '.']
trg = ['a', 'woman', 'with', 'a', 'large', 'purse', 'is', 'walking', 'by', 'a', 'gate', '.']


In [124]:
translation, attention = translate_sentence(src, SRC, TRG, model, device)

print(f'predicted trg = {translation}')

predicted trg = ['a', 'woman', 'with', 'a', 'large', 'purse', 'walks', 'past', 'a', 'gate', '.', '<eos>']


In [125]:
def translate_sentence_no_cache(sentence, src_field, trg_field, model, device, max_len=50):
    model.eval()

    tokens = src_field.preprocess(sentence)
    tokens = [src_field.init_token] + tokens + [src_field.eos_token]
    src_indexes = [src_field.vocab.stoi[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src_tensor)

    with torch.no_grad():
        enc_src = model.encoder(src_tensor, src_mask)

    trg_indexes = [trg_field.vocab.stoi[trg_field.init_token]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor(trg_indexes).unsqueeze(0).to(device)
        trg_mask = model.make_trg_mask(trg_tensor)

        with torch.no_grad():
            # past_kvs=None forces recomputation every step
            output, attention, _ = model.decoder(
                trg_tensor, enc_src, trg_mask, src_mask, past_kvs=None
            )

        pred_token = output.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)

        if pred_token == trg_field.vocab.stoi[trg_field.eos_token]:
            break

    trg_tokens = [trg_field.vocab.itos[i] for i in trg_indexes]
    return trg_tokens[1:], attention


def translate_sentence_with_cache(sentence, src_field, trg_field, model, device, max_len=50):
    model.eval()

    tokens = src_field.preprocess(sentence)
    tokens = [src_field.init_token] + tokens + [src_field.eos_token]
    src_indexes = [src_field.vocab.stoi[token] for token in tokens]
    src_tensor = torch.LongTensor(src_indexes).unsqueeze(0).to(device)
    src_mask = model.make_src_mask(src_tensor)

    with torch.no_grad():
        enc_src = model.encoder(src_tensor, src_mask)

    trg_indexes = [trg_field.vocab.stoi[trg_field.init_token]]
    past_kvs = None  # cache

    for i in range(max_len):
        # Only last token fed each step
        last_token = torch.LongTensor([trg_indexes[-1]]).unsqueeze(0).to(device)
        trg_mask = model.make_trg_mask(last_token)

        with torch.no_grad():
            output, attention, past_kvs = model.decoder(
                last_token, enc_src, trg_mask, src_mask, past_kvs
            )

        pred_token = output.argmax(2)[:, -1].item()
        trg_indexes.append(pred_token)

        if pred_token == trg_field.vocab.stoi[trg_field.eos_token]:
            break

    trg_tokens = [trg_field.vocab.itos[i] for i in trg_indexes]
    return trg_tokens[1:], attention

## Benchmark

Let's test the model without and with the cache KV in order to check the improvements. The speed up is not that significant, but using larger models have a lot more to compute compared to the current model.   

In [126]:
import time
import numpy as np


def benchmark(sentences, src_field, trg_field, model, device, max_len=50, runs=3):
    """
    Compare cache vs no-cache on:
    - Speed (ms per sentence)
    - Output equality (sanity check)
    """
    results = {
        "no_cache": {"times": [], "tokens": []},
        "cache":    {"times": [], "tokens": []},
    }

    print(f"{'Sentence':<45} {'No Cache':>12} {'Cache':>12} {'Speedup':>10} {'Match':>8}")
    print("-" * 95)

    for sentence in sentences:
        # No cache
        times_no_cache = []
        for _ in range(runs):
            start = time.perf_counter()
            tokens_no_cache, _ = translate_sentence_no_cache(
                sentence, src_field, trg_field, model, device, max_len
            )
            times_no_cache.append(time.perf_counter() - start)

        # With cache
        times_cache = []
        for _ in range(runs):
            start = time.perf_counter()
            tokens_cache, _ = translate_sentence_with_cache(
                sentence, src_field, trg_field, model, device, max_len
            )
            times_cache.append(time.perf_counter() - start)

        avg_no_cache = np.mean(times_no_cache) * 1000  # ms
        avg_cache = np.mean(times_cache)    * 1000
        speedup = avg_no_cache / avg_cache
        matched = tokens_no_cache == tokens_cache

        results["no_cache"]["times"].append(avg_no_cache)
        results["cache"]["times"].append(avg_cache)
        results["no_cache"]["tokens"].append(tokens_no_cache)
        results["cache"]["tokens"].append(tokens_cache)

        # Truncate sentence for display
        display = sentence[:42] + "..." if len(sentence) > 42 else sentence
        print(
            f"{display:<45} "
            f"{avg_no_cache:>10.1f}ms "
            f"{avg_cache:>10.1f}ms "
            f"{speedup:>9.2f}x "
            f"{'V' if matched else 'X':>8}"
        )

    # Summary
    total_no_cache = np.mean(results["no_cache"]["times"])
    total_cache    = np.mean(results["cache"]["times"])
    all_match      = all(
        a == b for a, b in zip(
            results["no_cache"]["tokens"],
            results["cache"]["tokens"]
        )
    )

    print("-" * 95)
    print(f"{'AVERAGE':<45} {total_no_cache:>10.1f}ms {total_cache:>10.1f}ms "
          f"{total_no_cache/total_cache:>9.2f}x {'V' if all_match else 'X':>8}")
    print(f"\nAll outputs identical: {'Yes' if all_match else 'No - bug in cache!'}")

    return results


test_sentences = [
    "A man is eating a piece of bread. A longer sentence could change the benchmark results for better visability.",
    "The girl is carrying a baby. A longer sentence could change the benchmark results for better visability.",
    "A man is riding a horse. A longer sentence could change the benchmark results for better visability.",
    "Two women are pushing children in strollers. A longer sentence could change the benchmark results for better visability.",
    "A man is playing guitar on the street and the weather is cold outside. A longer sentence could change the benchmark results for better visability.",
]

results = benchmark(
    test_sentences,
    SRC, TRG, model, device,
    max_len=50,
    runs=5
)

Sentence                                          No Cache        Cache    Speedup    Match
-----------------------------------------------------------------------------------------------
A man is eating a piece of bread. A longer...      114.2ms       97.7ms      1.17x        V
The girl is carrying a baby. A longer sent...       92.0ms       84.9ms      1.08x        V
A man is riding a horse. A longer sentence...       84.1ms       84.2ms      1.00x        V
Two women are pushing children in stroller...       84.3ms       88.1ms      0.96x        V
A man is playing guitar on the street and ...      183.0ms      174.3ms      1.05x        V
-----------------------------------------------------------------------------------------------
AVERAGE                                            111.5ms      105.9ms      1.05x        V

All outputs identical: Yes
